# PyPilot — PPO RL Training Loop

Loads the SFT'd LoRA checkpoint and runs PPO on top, optimizing for
functional correctness (compile + test pass) rather than next-token CE.

**Reward shaping (from spec):**
- `+1.0` passes all tests
- `+0.3` compiles but fails tests
- `-0.2` syntax / runtime error
- `-1.0` timeout

**Components:** PPO clipped objective + per-token advantage
(`A_t = R - V(s_t)`) + KL anchor to SFT model + SFT mix loss + pass@k
sampling + advantage normalization.

Set `LOCAL_TEST = True` for a smoke test on a single GPU (5070, 12 GB).


In [ ]:
# ════════════════════════════════════════════════════════════════════
# CONFIG — flip LOCAL_TEST to switch between smoke test and full run
# ════════════════════════════════════════════════════════════════════
LOCAL_TEST = True

if LOCAL_TEST:
    # Smoke test on RTX 5070 (12 GB). Tiny model, no quantization,
    # fresh LoRA (skip loading existing finetune), tiny batches.
    MODEL_ID         = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
    LORA_PATH        = None          # don't load existing finetune
    USE_4BIT         = False
    DATASET_NAME     = "newfacade/LeetCodeDataset"
    DATASET_SPLIT    = "train"
    NUM_PROBLEMS     = 8             # subset to cycle through
    BATCH_PROBLEMS   = 2             # prompts per rollout batch
    PASS_K           = 1             # rollouts per prompt
    MAX_NEW_TOKENS   = 256
    NUM_ITERATIONS   = 2             # outer RL iterations
    PPO_EPOCHS       = 1             # inner PPO update epochs per iteration
    PPO_MINIBATCH    = 2
    OUTPUT_DIR       = "./pypilot_rl_local"
else:
    # Full training. Load existing SFT'd LoRA, 4-bit base, pass@k sampling.
    MODEL_ID         = "Qwen/Qwen3.5-9B"
    LORA_PATH        = "/outputs/final-20260429T192926Z-3-001/final"
    USE_4BIT         = True
    DATASET_NAME     = "/data/leetcode_clean-20260429T192835Z-3-001/leetcode_clean"
    DATASET_SPLIT    = "train"
    NUM_PROBLEMS     = None          # full split
    BATCH_PROBLEMS   = 4
    PASS_K           = 4
    MAX_NEW_TOKENS   = 1024
    NUM_ITERATIONS   = 100
    PPO_EPOCHS       = 4
    PPO_MINIBATCH    = 4
    OUTPUT_DIR       = "/outputs/qwen-lora-rl"

# ─── PPO hyperparameters (same in both modes) ─────────────────────────
LR_POLICY        = 1e-5
LR_VALUE         = 1e-4
CLIP_EPS         = 0.2
KL_COEF          = 0.05      # weight on KL(policy || ref_SFT) penalty
SFT_COEF         = 0.1       # weight on SFT mix loss
VALUE_COEF       = 0.5
ADV_NORMALIZE    = True

# ─── Reward shaping ───────────────────────────────────────────────────
R_PASS           =  1.0
R_COMPILE_FAIL   =  0.3
R_SYNTAX_ERR     = -0.2
R_TIMEOUT        = -1.0
TEST_TIMEOUT_SEC = 10.0

print(f"Mode: {'LOCAL_TEST' if LOCAL_TEST else 'FULL'}")
print(f"Model: {MODEL_ID}  |  LoRA path: {LORA_PATH}  |  4-bit: {USE_4BIT}")


Mode: LOCAL_TEST
Model: Qwen/Qwen2.5-Coder-0.5B-Instruct  |  LoRA path: None  |  4-bit: False


In [ ]:
import os, sys, gc, ast, json, subprocess, tempfile, random, re
from pathlib import Path
from contextlib import contextmanager

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

from datasets import load_dataset, load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
print(f"Device: {DEVICE}, dtype: {DTYPE}")

torch.manual_seed(0); random.seed(0); np.random.seed(0)


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Tokenizer + ChatML prompt template
# (matches the SFT pipeline exactly — same SYSTEM_PROMPT and tokens)
# ════════════════════════════════════════════════════════════════════
SYSTEM_PROMPT = (
    "You are a competitive programming expert. "
    "Given a problem description and starter code, write a correct Python solution. "
    "Output only the Python code with no explanations or markdown."
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    # Use the dedicated FIM pad token, not eos — same fix as the SFT script
    tokenizer.pad_token = "<|fim_pad|>" if "<|fim_pad|>" in tokenizer.get_vocab() else tokenizer.eos_token
tokenizer.padding_side = "left"   # left-pad for generation

def build_prompt(problem_text: str) -> str:
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{problem_text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Load policy = base + LoRA (existing or fresh) + value head.
# Reference policy = same model with adapters disabled (saves a 2nd copy).
# ════════════════════════════════════════════════════════════════════
bnb_cfg = None
if USE_4BIT:
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=DTYPE,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    torch_dtype=DTYPE,
    device_map={"": DEVICE},
    trust_remote_code=True,
)
if USE_4BIT:
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
else:
    base.gradient_checkpointing_enable()
    base.enable_input_require_grads()

# Attach LoRA — load existing if provided, else fresh adapter
if LORA_PATH and Path(LORA_PATH).exists():
    print(f"Loading existing LoRA from {LORA_PATH}")
    policy = PeftModel.from_pretrained(base, LORA_PATH, is_trainable=True)
else:
    print("Creating fresh LoRA adapter")
    lora_cfg = LoraConfig(
        r=16, lora_alpha=16, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM", bias="none",
    )
    policy = get_peft_model(base, lora_cfg)

policy.print_trainable_parameters()

# Value head: linear from hidden_size → 1, kept in fp32 for stability
hidden_size = base.config.hidden_size
value_head = nn.Linear(hidden_size, 1, bias=True).to(DEVICE).to(torch.float32)
nn.init.normal_(value_head.weight, std=1.0 / (hidden_size + 1) ** 0.5)
nn.init.zeros_(value_head.bias)

# Optimizers — separate so policy and value can have different LRs
policy_params = [p for p in policy.parameters() if p.requires_grad]
optim_policy = AdamW(policy_params, lr=LR_POLICY)
optim_value  = AdamW(value_head.parameters(), lr=LR_VALUE)

print(f"Policy trainable params: {sum(p.numel() for p in policy_params):,}")
print(f"Value head params: {sum(p.numel() for p in value_head.parameters()):,}")


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Dataset — load LeetCode problems with their test harnesses
# ════════════════════════════════════════════════════════════════════
def load_leetcode(name, split):
    if Path(name).exists():
        ds = load_from_disk(name)
        return ds[split] if hasattr(ds, "keys") else ds
    return load_dataset(name, split=split)

raw = load_leetcode(DATASET_NAME, DATASET_SPLIT)
if NUM_PROBLEMS is not None:
    raw = raw.select(range(min(NUM_PROBLEMS, len(raw))))

# Normalize to (prompt_text, test_code, entry_point, reference_completion)
def normalize(ex):
    return {
        "prompt_text":   ex.get("prompt") or ex.get("problem") or "",
        "test":          ex.get("test", ""),
        "entry_point":   ex.get("entry_point") or ex.get("func_name") or "",
        "completion":    ex.get("completion") or ex.get("canonical_solution") or "",
    }
problems = [normalize(ex) for ex in raw]
print(f"Loaded {len(problems)} problems. Example fields: {list(problems[0].keys())}")
print(f"Entry point of first: {problems[0]['entry_point']}")


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Code extraction — slimmed copy of the eval-harness cleaner
# Handles ChatML token leakage, markdown fences, missing imports
# ════════════════════════════════════════════════════════════════════
_CHATML_TOKENS = (
    "<|im_start|>system", "<|im_start|>user", "<|im_start|>assistant",
    "<|im_start|>", "<|im_end|>", "<|endoftext|>",
)
_STOP_TOKENS = (
    "<|file_sep|>", "<|fim_prefix|>", "<|fim_suffix|>", "<|fim_middle|>",
    "<|repo_name|>",
)

def extract_code(completion: str) -> str:
    for t in _CHATML_TOKENS:
        completion = completion.replace(t, "")
    for t in _STOP_TOKENS:
        i = completion.find(t)
        if i != -1: completion = completion[:i]

    if "```python" in completion:
        s = completion.find("```python") + len("```python")
        e = completion.find("```", s)
        completion = completion[s:e] if e != -1 else completion[s:]
    elif "```" in completion:
        s = completion.find("```") + 3
        e = completion.find("```", s)
        completion = completion[s:e] if e != -1 else completion[s:]

    code = completion.strip()

    # Auto-add common missing imports
    typing_needed = [n for n in ("List", "Dict", "Tuple", "Optional", "Union", "Set")
                     if f"{n}[" in code]
    if re.search(r"\bAny\b", code): typing_needed.append("Any")
    if typing_needed and "from typing import" not in code:
        code = f"from typing import {', '.join(typing_needed)}\n" + code
    if "defaultdict" in code and "from collections" not in code:
        code = "from collections import defaultdict, deque, Counter\n" + code
    elif "deque" in code and "from collections" not in code:
        code = "from collections import deque\n" + code
    elif "Counter" in code and "from collections" not in code:
        code = "from collections import Counter\n" + code
    if ("heappush" in code or "heappop" in code) and "heapq" not in code:
        code = "from heapq import heappush, heappop, heapify\n" + code
    return code


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Reward — subprocess sandbox, returns scalar per spec
# ════════════════════════════════════════════════════════════════════
def evaluate_code(code_str: str, test_code: str, entry_point: str,
                  timeout: float = TEST_TIMEOUT_SEC):
    """Returns (reward, status_label)."""
    # 1. Syntax check (free, no subprocess)
    try:
        ast.parse(code_str)
    except SyntaxError:
        return R_SYNTAX_ERR, "syntax_error"

    full_script = f"{code_str}\n\n{test_code}\n\ncheck({entry_point})\n"
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(full_script)
        path = f.name
    try:
        result = subprocess.run(
            [sys.executable, path],
            capture_output=True, timeout=timeout, text=True,
        )
        if result.returncode == 0:
            return R_PASS, "passed"
        err = (result.stderr or "")
        if "SyntaxError" in err or "IndentationError" in err:
            return R_SYNTAX_ERR, "syntax_error"
        return R_COMPILE_FAIL, "test_failed"
    except subprocess.TimeoutExpired:
        return R_TIMEOUT, "timeout"
    except Exception:
        return R_SYNTAX_ERR, "exec_error"
    finally:
        try: os.unlink(path)
        except OSError: pass


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Generation (no-grad) + forward pass that returns per-token logprobs
# and per-token values for the response portion of the sequence.
# ════════════════════════════════════════════════════════════════════
@contextmanager
def _nullctx():
    yield

@torch.no_grad()
def generate_response(prompt_text: str, max_new_tokens: int = MAX_NEW_TOKENS):
    """Generate one completion. Returns full_ids, prompt_len, response_text."""
    enc = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=2048)
    input_ids = enc.input_ids.to(DEVICE)
    attn      = enc.attention_mask.to(DEVICE)
    prompt_len = input_ids.shape[1]

    out = policy.generate(
        input_ids=input_ids,
        attention_mask=attn,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    full_ids = out  # (1, prompt_len + new_tokens)
    response_text = tokenizer.decode(full_ids[0, prompt_len:], skip_special_tokens=False)
    return full_ids, prompt_len, response_text


def forward_logprobs_values(full_ids: torch.Tensor, prompt_len: int,
                            use_reference: bool = False):
    """
    Run a forward pass over full_ids and return logprobs of each *response*
    token, plus values at each response position. If use_reference=True,
    LoRA adapters are disabled so we get the SFT (anchor) policy.
    """
    ctx = policy.disable_adapter() if use_reference else _nullctx()
    with ctx:
        outputs = policy(
            input_ids=full_ids,
            attention_mask=torch.ones_like(full_ids),
            output_hidden_states=True,
            use_cache=False,
        )
    logits = outputs.logits                                                 # (B, T, V)
    # logprob of token at position t comes from logits[:, t-1]
    logp_full = F.log_softmax(logits[:, :-1].float(), dim=-1)
    target    = full_ids[:, 1:]
    token_logprobs = logp_full.gather(-1, target.unsqueeze(-1)).squeeze(-1) # (B, T-1)

    # Slice to response tokens only.
    # Response tokens occupy positions [prompt_len .. T-1] in full_ids,
    # which corresponds to indices [prompt_len-1 .. T-2] in token_logprobs.
    resp_logprobs = token_logprobs[:, prompt_len - 1:]                      # (B, R)

    # Values at the *state* before each response token's emission.
    # Hidden state at position t represents state after consuming token t.
    last_hidden = outputs.hidden_states[-1]                                 # (B, T, H)
    resp_values = value_head(last_hidden[:, prompt_len - 1:-1].float()).squeeze(-1)  # (B, R)

    return resp_logprobs, resp_values, logits


In [ ]:
# ════════════════════════════════════════════════════════════════════
# PPO loss components
#   - per-token advantage A_t = R - V(s_t)  (per spec)
#   - clipped surrogate objective
#   - clipped value loss
#   - KL anchor to reference (SFT) policy
#   - SFT mix loss on reference completions
# ════════════════════════════════════════════════════════════════════
def compute_advantages(rewards, values, mask):
    """
    rewards: (B,)         scalar terminal reward per trajectory
    values:  (B, R)       per-token value estimates
    mask:    (B, R)       1 for real response tokens, 0 for padding
    Returns advantages (B, R), returns (B, R)
    """
    returns    = rewards.unsqueeze(1).expand_as(values) * mask
    advantages = (returns - values) * mask
    if ADV_NORMALIZE:
        flat = advantages[mask.bool()]
        if flat.numel() > 1:
            advantages = (advantages - flat.mean()) / (flat.std() + 1e-8)
            advantages = advantages * mask
    return advantages.detach(), returns.detach()


def ppo_losses(new_logprobs, old_logprobs, ref_logprobs,
               new_values, old_values, returns,
               advantages, mask):
    """All loss terms, masked over response tokens."""
    m = mask.float()
    n = m.sum().clamp(min=1.0)

    # 1. Clipped policy loss
    ratio = torch.exp(new_logprobs - old_logprobs)
    s1 = ratio * advantages
    s2 = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * advantages
    pg_loss = -torch.min(s1, s2)
    pg_loss = (pg_loss * m).sum() / n

    # 2. Clipped value loss
    v_clipped = old_values + torch.clamp(new_values - old_values, -CLIP_EPS, CLIP_EPS)
    v_l = torch.max((new_values - returns) ** 2, (v_clipped - returns) ** 2)
    v_loss = 0.5 * (v_l * m).sum() / n

    # 3. KL anchor (current vs reference SFT)
    kl = (new_logprobs - ref_logprobs)
    kl_loss = (kl * m).sum() / n

    # Diagnostic clip fraction
    clip_frac = ((torch.abs(ratio - 1.0) > CLIP_EPS).float() * m).sum() / n

    return pg_loss, v_loss, kl_loss, clip_frac


def sft_mix_loss(prompt_text: str, reference_completion: str):
    """Cross-entropy on the SFT reference completion — keeps syntax intact."""
    full = prompt_text + reference_completion + "<|im_end|>"
    enc = tokenizer(full, return_tensors="pt", truncation=True, max_length=2048).to(DEVICE)
    prompt_len = tokenizer(prompt_text, return_tensors="pt").input_ids.shape[1]
    labels = enc.input_ids.clone()
    labels[:, :prompt_len] = -100  # mask prompt
    out = policy(input_ids=enc.input_ids, attention_mask=enc.attention_mask, labels=labels)
    return out.loss


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Main RL training loop
# ════════════════════════════════════════════════════════════════════
def rollout_one(problem):
    """pass@k sampling for one problem. Returns the best (highest-reward) trajectory."""
    prompt = build_prompt(problem["prompt_text"])
    best = None
    for _ in range(PASS_K):
        full_ids, prompt_len, response_text = generate_response(prompt)
        code_str = extract_code(response_text)
        reward, status = evaluate_code(code_str, problem["test"], problem["entry_point"])
        traj = dict(prompt=prompt, full_ids=full_ids, prompt_len=prompt_len,
                    code=code_str, reward=reward, status=status,
                    reference_completion=problem["completion"])
        if best is None or reward > best["reward"]:
            best = traj
    return best


def collect_rollouts(batch):
    return [rollout_one(p) for p in batch]


def ppo_update(rollouts):
    """Run PPO_EPOCHS of updates over the collected rollouts."""
    # ── Pre-compute "old" logprobs/values + reference logprobs (no grad) ──
    olds = []
    with torch.no_grad():
        for tr in rollouts:
            old_lp, old_v, _ = forward_logprobs_values(tr["full_ids"], tr["prompt_len"], use_reference=False)
            ref_lp, _,    _ = forward_logprobs_values(tr["full_ids"], tr["prompt_len"], use_reference=True)
            olds.append({"old_lp": old_lp[0].detach(), "old_v": old_v[0].detach(),
                         "ref_lp": ref_lp[0].detach()})

    metrics = {"pg": [], "v": [], "kl": [], "sft": [], "clip_frac": []}
    idxs = list(range(len(rollouts)))
    for _ in range(PPO_EPOCHS):
        random.shuffle(idxs)
        for start in range(0, len(idxs), PPO_MINIBATCH):
            mb = idxs[start:start + PPO_MINIBATCH]
            optim_policy.zero_grad(set_to_none=True)
            optim_value.zero_grad(set_to_none=True)

            mb_pg = mb_v = mb_kl = mb_sft = mb_cf = 0.0
            for i in mb:
                tr = rollouts[i]
                old = olds[i]
                # Fresh forward — gradients on
                new_lp, new_v, _ = forward_logprobs_values(tr["full_ids"], tr["prompt_len"])
                new_lp = new_lp[0]; new_v = new_v[0]

                R = torch.tensor(tr["reward"], device=DEVICE, dtype=torch.float32)
                mask = torch.ones_like(new_lp)
                adv, ret = compute_advantages(R.unsqueeze(0),
                                              old["old_v"].unsqueeze(0),
                                              mask.unsqueeze(0))
                adv = adv[0]; ret = ret[0]

                pg, vloss, klloss, cf = ppo_losses(
                    new_lp, old["old_lp"], old["ref_lp"],
                    new_v,  old["old_v"],  ret,
                    adv,    mask,
                )
                if tr["reference_completion"]:
                    sft = sft_mix_loss(tr["prompt"], tr["reference_completion"])
                else:
                    sft = torch.tensor(0.0, device=DEVICE)

                loss = pg + VALUE_COEF * vloss + KL_COEF * klloss + SFT_COEF * sft
                loss = loss / len(mb)
                loss.backward()

                mb_pg  += pg.item()  / len(mb)
                mb_v   += vloss.item() / len(mb)
                mb_kl  += klloss.item() / len(mb)
                mb_sft += float(sft.item()) / len(mb)
                mb_cf  += cf.item() / len(mb)

            torch.nn.utils.clip_grad_norm_(policy_params, 1.0)
            torch.nn.utils.clip_grad_norm_(value_head.parameters(), 1.0)
            optim_policy.step()
            optim_value.step()

            metrics["pg"].append(mb_pg);   metrics["v"].append(mb_v)
            metrics["kl"].append(mb_kl);   metrics["sft"].append(mb_sft)
            metrics["clip_frac"].append(mb_cf)

    return {k: float(np.mean(v)) if v else 0.0 for k, v in metrics.items()}


# ── Run ───────────────────────────────────────────────────────────────
history = []
for it in range(NUM_ITERATIONS):
    batch = random.sample(problems, min(BATCH_PROBLEMS, len(problems)))
    rollouts = collect_rollouts(batch)

    rewards = [r["reward"] for r in rollouts]
    statuses = [r["status"] for r in rollouts]
    pass_rate = sum(1 for s in statuses if s == "passed") / len(statuses)

    update_metrics = ppo_update(rollouts)

    log = {"iter": it,
           "mean_reward": float(np.mean(rewards)),
           "pass_rate": pass_rate,
           **update_metrics}
    history.append(log)
    print(f"[iter {it:03d}] R={log['mean_reward']:+.3f}  pass={pass_rate:.2%}  "
          f"pg={log['pg']:+.4f}  v={log['v']:.4f}  kl={log['kl']:+.4f}  "
          f"sft={log['sft']:.3f}  clip={log['clip_frac']:.2f}")

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Save: LoRA adapter (PEFT) + value head + training history
# ════════════════════════════════════════════════════════════════════
out = Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
policy.save_pretrained(out / "lora_rl")
torch.save(value_head.state_dict(), out / "value_head.pt")
with open(out / "rl_history.json", "w") as f:
    json.dump(history, f, indent=2)
print(f"Saved to {out}")
